# Short-Term Solar Power Forecasting (1 Hour Ahead)

**Goal:** predict the AC power output of a solar plant **one hour ahead** using recent power readings and weather sensor data.

**Why it matters:** accurate short-term solar forecasts help grid operators balance supply and demand, plan storage and reduce reliance on backup generation. This is a core flexibility problem for a grid with a high share of renewables.

**Data:** Solar Power Generation Data (Kaggle, by Ani Kannal): 2 plants in India, 34 days, 15-min resolution. Plant 1 is used here.

**Approach:**
1. Load and clean the data, then merge generation with weather.
2. Explore the relationship between irradiation and power.
3. Build lag and time features.
4. Split by time (no random shuffling) and compare XGBoost with two simple baselines.
5. Evaluate with MAE and RMSE and inspect the results.

## 1. Imports and settings

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Forecast settings
STEP_MINUTES = 15          # data resolution
HORIZON_STEPS = 4          # 4 steps x 15 min = 1 hour ahead
TEST_DAYS = 6              # last 6 days are held out for testing
RANDOM_STATE = 42

plt.rcParams["figure.figsize"] = (12, 4)

## 2. Load the data

The generation file has one row per **inverter** per timestamp. The weather file has one row per timestamp for the whole plant. Note that the two files use **different date formats** (`15-05-2020 00:00` vs `2020-05-15 00:00:00`), so they are parsed separately.

In [ ]:
# Find the dataset folder automatically (the Kaggle path can differ between versions)
DATA_ROOT = "/kaggle/input"
gen_file = glob.glob(f"{DATA_ROOT}/**/Plant_1_Generation_Data.csv", recursive=True)[0]
P = os.path.dirname(gen_file) + "/"

gen = pd.read_csv(P + "Plant_1_Generation_Data.csv")
wea = pd.read_csv(P + "Plant_1_Weather_Sensor_Data.csv")

print("Generation:", gen.shape, "| Weather:", wea.shape)
gen.head(3)

## 3. Clean and merge

Steps:
- Parse timestamps with the correct format for each file.
- **Sum AC power across all inverters** to get total plant output per timestamp.
- Join with the weather readings on the timestamp.
- Check for missing timestamps and fill short gaps (up to 1 hour) by interpolation.

In [ ]:
gen["DATE_TIME"] = pd.to_datetime(gen["DATE_TIME"], format="%d-%m-%Y %H:%M")
wea["DATE_TIME"] = pd.to_datetime(wea["DATE_TIME"])

plant_power = gen.groupby("DATE_TIME")["AC_POWER"].sum()

df = (
    wea.set_index("DATE_TIME")[["AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]]
    .join(plant_power, how="inner")
    .sort_index()
)

# Check for missing 15-minute timestamps and fill short gaps
full_index = pd.date_range(df.index.min(), df.index.max(), freq=f"{STEP_MINUTES}min")
print(f"Rows: {len(df)} | Expected: {len(full_index)} | Missing timestamps: {len(full_index) - len(df)}")

df = df.reindex(full_index).interpolate(limit=4)
print("Remaining NaNs after interpolation:", int(df.isna().sum().sum()))
df.describe().round(2)

## 4. Exploratory analysis

Two quick checks before modelling: how power behaves over a few days, and how strongly it depends on solar irradiation.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))

df["AC_POWER"].iloc[:96 * 4].plot(ax=ax[0])
ax[0].set_title("Plant AC power, first 4 days")
ax[0].set_ylabel("AC power (kW)")

ax[1].scatter(df["IRRADIATION"], df["AC_POWER"], s=4, alpha=0.4)
ax[1].set_title("AC power vs irradiation")
ax[1].set_xlabel("Irradiation")
ax[1].set_ylabel("AC power (kW)")

plt.tight_layout()
plt.show()

print("Correlation of AC power with:")
print(df.corr()["AC_POWER"].drop("AC_POWER").round(3))

## 5. Feature engineering

The target is the plant's AC power **1 hour in the future**. Features available at prediction time:

| Feature | Meaning |
|---|---|
| `hour` | time of day as a decimal (captures the daily solar cycle) |
| `AMBIENT_TEMPERATURE`, `MODULE_TEMPERATURE`, `IRRADIATION` | current weather readings |
| `AC_POWER` | current power output |
| `power_lag1`, `power_lag4` | power 15 min and 1 hour ago (recent trend) |
| `irr_lag4` | irradiation 1 hour ago |

`yesterday` is **not** used by the model. It only feeds the "same time yesterday" baseline.

In [ ]:
H = HORIZON_STEPS

df["hour"] = df.index.hour + df.index.minute / 60
df["power_lag1"] = df["AC_POWER"].shift(1)
df["power_lag4"] = df["AC_POWER"].shift(4)
df["irr_lag4"] = df["IRRADIATION"].shift(4)

# Target: power H steps in the future
df["target"] = df["AC_POWER"].shift(-H)

# Baseline helper: the value at the same clock time as the target, one day earlier
df["yesterday"] = df["AC_POWER"].shift(24 * 60 // STEP_MINUTES - H)

FEATURES = [
    "hour", "AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION",
    "AC_POWER", "power_lag1", "power_lag4", "irr_lag4",
]

data = df.dropna(subset=FEATURES + ["target", "yesterday"])
print("Usable rows:", len(data))
data[FEATURES + ["target"]].head()

## 6. Train / test split (by time)

Time series must **not** be split randomly, otherwise the model sees the future during training. The **last 6 days** are held out as the test set.

In [ ]:
cutoff = data.index.max() - pd.Timedelta(days=TEST_DAYS)
train = data[data.index <= cutoff]
test = data[data.index > cutoff]

print(f"Train: {train.index.min()} -> {train.index.max()}  ({len(train)} rows)")
print(f"Test : {test.index.min()} -> {test.index.max()}  ({len(test)} rows)")

## 7. Baselines and model

Two simple baselines give the model something to beat:
- **Persistence:** the power one hour from now equals the power right now.
- **Same time yesterday:** the power one hour from now equals yesterday's power at that clock time.

The model is an **XGBoost regressor** trained on the features above.

In [ ]:
def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {"Model": name, "MAE": mae, "RMSE": rmse}

y_test = test["target"]

results = [
    evaluate("Baseline: persistence", y_test, test["AC_POWER"]),
    evaluate("Baseline: same time yesterday", y_test, test["yesterday"]),
]

model = XGBRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE
)
model.fit(train[FEATURES], train["target"])
pred = model.predict(test[FEATURES])
pred = np.clip(pred, 0, None)   # power cannot be negative

results.append(evaluate("XGBoost", y_test, pred))
results_df = pd.DataFrame(results).set_index("Model").round(1)
results_df

## 8. Results

RMSE improvement of XGBoost over each baseline (positive means XGBoost is better, negative means it is worse):

In [ ]:
rmse = results_df["RMSE"]
for base in ["Baseline: persistence", "Baseline: same time yesterday"]:
    gain = (1 - rmse["XGBoost"] / rmse[base]) * 100
    print(f"XGBoost vs {base:32s}: {gain:+.1f}% RMSE")

In [ ]:
plt.figure()
plt.plot(test.index, y_test, label="Actual", lw=1.2)
plt.plot(test.index, pred, label="XGBoost (1h ahead)", lw=1.2)
plt.title("Solar AC power: actual vs 1-hour-ahead forecast (test set)")
plt.ylabel("AC power (kW)")
plt.legend()
plt.tight_layout()
plt.savefig("forecast.png", dpi=150)
plt.show()

### Interpretation

- XGBoost reduced RMSE by **32.1%** versus the persistence baseline and by **26.0%** versus the same-time-yesterday baseline on a held-out 6-day test set.
- The forecast follows the daily solar cycle closely. It lags behind abrupt changes (likely passing clouds), which is expected because the model only sees current and past readings, not the weather of the next hour.

## 9. What drives the forecast?

Feature importance from the trained model shows which inputs it relies on most.

In [ ]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
imp.plot(kind="barh", figsize=(8, 4), title="XGBoost feature importance")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

### Interpretation

Current AC power (about 0.51) and irradiation (about 0.33) carry most of the model's predictive weight, with time of day third (about 0.09). Lag features and temperatures add little. Importance values describe what the model relies on, not causal effects.

## 10. Limitations and next steps

- **Only 34 days of data:** the test set is 6 days, so results can vary with the weather of those specific days. Treat the numbers as indicative, not as a general benchmark.
- **One plant, one season:** the model has not seen other seasons or climates.
- **Next steps:** test generalisation on **Plant 2**; add a sequence model (LSTM) and compare; try other horizons (15 min, 3 hours); use a numerical weather forecast as an input instead of current readings.